<a href="https://colab.research.google.com/github/jennyk23/Magpy/blob/main/NomeSobrenome_Sakila_SQL_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Sakila PT-BR — execução completa no Google Colab

Este notebook:
1. instala e inicia o MariaDB;
2. baixa e importa a base oficial Sakila;
3. cria a base `locadora` com tabelas e colunas em português;
4. executa todos os exercícios, procedures e triggers;
5. permite baixar o arquivo SQL final.

**Execute as células na ordem.**

In [1]:
import os
import shutil
import subprocess
import time

def executar(comando, *, entrada=None, verificar=True):
    resultado = subprocess.run(
        comando,
        stdin=entrada,
        text=True,
        capture_output=True
    )
    if verificar and resultado.returncode != 0:
        print(resultado.stdout)
        print(resultado.stderr)
        raise RuntimeError(
            "Falha ao executar: " + " ".join(comando)
        )
    return resultado

if shutil.which("mariadb") is None:
    print("Instalando MariaDB e utilitários...")
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True
    )
    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "mariadb-server",
            "mariadb-client",
            "unzip"
        ],
        check=True,
        stdout=subprocess.DEVNULL
    )

os.makedirs("/run/mysqld", exist_ok=True)
subprocess.run(
    ["chown", "mysql:mysql", "/run/mysqld"],
    check=False
)
subprocess.run(
    ["service", "mariadb", "start"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    teste = executar(
        [
            "mariadb-admin",
            "-u",
            "root",
            "ping",
            "--silent"
        ],
        verificar=False
    )
    if teste.returncode == 0:
        break
    time.sleep(1)
else:
    raise RuntimeError(
        "Não foi possível iniciar o MariaDB."
    )

print("MariaDB iniciado com sucesso.")
print(
    executar(
        ["mariadb", "--version"]
    ).stdout
)


Instalando MariaDB e utilitários...
MariaDB iniciado com sucesso.
mariadb  Ver 15.1 Distrib 10.6.23-MariaDB, for debian-linux-gnu (x86_64) using  EditLine wrapper



In [2]:
from pathlib import Path
from urllib.request import urlretrieve
import zipfile
import subprocess

url = "https://downloads.mysql.com/docs/sakila-db.zip"
zip_path = Path("/content/sakila-db.zip")
extract_dir = Path("/content/sakila-db-extraido")

print("Baixando a base oficial Sakila...")
urlretrieve(url, zip_path)

extract_dir.mkdir(
    parents=True,
    exist_ok=True
)

with zipfile.ZipFile(zip_path, "r") as arquivo_zip:
    arquivo_zip.extractall(extract_dir)

schema_files = list(
    extract_dir.rglob("sakila-schema.sql")
)
data_files = list(
    extract_dir.rglob("sakila-data.sql")
)

if not schema_files or not data_files:
    raise FileNotFoundError(
        "Os arquivos oficiais da base Sakila não foram encontrados."
    )

schema_path = schema_files[0]
data_path = data_files[0]

print("Importando a estrutura oficial...")
with schema_path.open("r", encoding="utf-8", errors="replace") as entrada:
    resultado = subprocess.run(
        [
            "mariadb",
            "-u",
            "root",
            "--default-character-set=utf8mb4"
        ],
        stdin=entrada,
        text=True,
        capture_output=True
    )

if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError(
        "Falha ao importar sakila-schema.sql."
    )

print("Importando os dados oficiais...")
with data_path.open("r", encoding="utf-8", errors="replace") as entrada:
    resultado = subprocess.run(
        [
            "mariadb",
            "-u",
            "root",
            "--default-character-set=utf8mb4"
        ],
        stdin=entrada,
        text=True,
        capture_output=True
    )

if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError(
        "Falha ao importar sakila-data.sql."
    )

print("Base oficial sakila importada com sucesso.")


Baixando a base oficial Sakila...
Importando a estrutura oficial...
Importando os dados oficiais...
Base oficial sakila importada com sucesso.


In [3]:
from pathlib import Path
import subprocess

setup_sql = "\n-- ============================================================\n-- PREPARAÇÃO DA BASE SAKILA PT-BR PARA A ATIVIDADE\n-- Origem dos dados: banco oficial sakila\n-- Banco usado nos exercícios: locadora\n-- ============================================================\n\nDROP DATABASE IF EXISTS locadora;\n\nCREATE DATABASE locadora\n    CHARACTER SET utf8mb4\n    COLLATE utf8mb4_unicode_ci;\n\nUSE locadora;\n\nCREATE TABLE pais (\n    pais_id SMALLINT UNSIGNED NOT NULL,\n    pais VARCHAR(50) NOT NULL,\n    PRIMARY KEY (pais_id)\n) ENGINE = InnoDB;\n\nCREATE TABLE cidade (\n    cidade_id SMALLINT UNSIGNED NOT NULL,\n    cidade VARCHAR(50) NOT NULL,\n    pais_id SMALLINT UNSIGNED NOT NULL,\n    PRIMARY KEY (cidade_id),\n    CONSTRAINT fk_cidade_pais\n        FOREIGN KEY (pais_id)\n        REFERENCES pais(pais_id)\n) ENGINE = InnoDB;\n\nCREATE TABLE endereco (\n    endereco_id SMALLINT UNSIGNED NOT NULL,\n    endereco VARCHAR(100) NOT NULL,\n    cidade_id SMALLINT UNSIGNED NOT NULL,\n    PRIMARY KEY (endereco_id),\n    CONSTRAINT fk_endereco_cidade\n        FOREIGN KEY (cidade_id)\n        REFERENCES cidade(cidade_id)\n) ENGINE = InnoDB;\n\nCREATE TABLE cliente (\n    cliente_id SMALLINT UNSIGNED NOT NULL,\n    primeiro_nome VARCHAR(45) NOT NULL,\n    ultimo_nome VARCHAR(45) NOT NULL,\n    email VARCHAR(50),\n    endereco_id SMALLINT UNSIGNED NOT NULL,\n    PRIMARY KEY (cliente_id),\n    CONSTRAINT fk_cliente_endereco\n        FOREIGN KEY (endereco_id)\n        REFERENCES endereco(endereco_id)\n) ENGINE = InnoDB;\n\nCREATE TABLE ator (\n    ator_id SMALLINT UNSIGNED NOT NULL,\n    primeiro_nome VARCHAR(45) NOT NULL,\n    ultimo_nome VARCHAR(45) NOT NULL,\n    PRIMARY KEY (ator_id)\n) ENGINE = InnoDB;\n\nCREATE TABLE categoria (\n    categoria_id TINYINT UNSIGNED NOT NULL,\n    nome VARCHAR(25) NOT NULL,\n    PRIMARY KEY (categoria_id)\n) ENGINE = InnoDB;\n\nCREATE TABLE filme (\n    filme_id SMALLINT UNSIGNED NOT NULL AUTO_INCREMENT,\n    titulo VARCHAR(128) NOT NULL,\n    descricao TEXT,\n    ano_lancamento YEAR,\n    idioma_id TINYINT UNSIGNED NOT NULL,\n    duracao_locacao TINYINT UNSIGNED NOT NULL DEFAULT 3,\n    preco_da_locacao DECIMAL(4,2) NOT NULL DEFAULT 4.99,\n    duracao_do_filme SMALLINT UNSIGNED,\n    custo_substituicao DECIMAL(5,2) NOT NULL DEFAULT 19.99,\n    classificacao VARCHAR(10),\n    PRIMARY KEY (filme_id),\n    CONSTRAINT chk_filme_preco_nao_negativo\n        CHECK (preco_da_locacao >= 0)\n) ENGINE = InnoDB;\n\nCREATE TABLE filme_categoria (\n    filme_id SMALLINT UNSIGNED NOT NULL,\n    categoria_id TINYINT UNSIGNED NOT NULL,\n    PRIMARY KEY (filme_id, categoria_id),\n    CONSTRAINT fk_filme_categoria_filme\n        FOREIGN KEY (filme_id)\n        REFERENCES filme(filme_id)\n        ON DELETE CASCADE,\n    CONSTRAINT fk_filme_categoria_categoria\n        FOREIGN KEY (categoria_id)\n        REFERENCES categoria(categoria_id)\n) ENGINE = InnoDB;\n\nCREATE TABLE filme_ator (\n    ator_id SMALLINT UNSIGNED NOT NULL,\n    filme_id SMALLINT UNSIGNED NOT NULL,\n    PRIMARY KEY (ator_id, filme_id),\n    CONSTRAINT fk_filme_ator_ator\n        FOREIGN KEY (ator_id)\n        REFERENCES ator(ator_id),\n    CONSTRAINT fk_filme_ator_filme\n        FOREIGN KEY (filme_id)\n        REFERENCES filme(filme_id)\n        ON DELETE CASCADE\n) ENGINE = InnoDB;\n\nCREATE TABLE inventario (\n    inventario_id MEDIUMINT UNSIGNED NOT NULL,\n    filme_id SMALLINT UNSIGNED NOT NULL,\n    PRIMARY KEY (inventario_id),\n    CONSTRAINT fk_inventario_filme\n        FOREIGN KEY (filme_id)\n        REFERENCES filme(filme_id)\n) ENGINE = InnoDB;\n\nCREATE TABLE aluguel (\n    aluguel_id INT NOT NULL,\n    data_aluguel DATETIME NOT NULL,\n    inventario_id MEDIUMINT UNSIGNED NOT NULL,\n    cliente_id SMALLINT UNSIGNED NOT NULL,\n    data_devolucao DATETIME,\n    PRIMARY KEY (aluguel_id),\n    CONSTRAINT fk_aluguel_inventario\n        FOREIGN KEY (inventario_id)\n        REFERENCES inventario(inventario_id),\n    CONSTRAINT fk_aluguel_cliente\n        FOREIGN KEY (cliente_id)\n        REFERENCES cliente(cliente_id)\n) ENGINE = InnoDB;\n\nCREATE TABLE pagamento (\n    pagamento_id SMALLINT UNSIGNED NOT NULL AUTO_INCREMENT,\n    cliente_id SMALLINT UNSIGNED NOT NULL,\n    aluguel_id INT,\n    valor DECIMAL(5,2) NOT NULL,\n    data_pagamento DATETIME NOT NULL,\n    PRIMARY KEY (pagamento_id),\n    CONSTRAINT fk_pagamento_cliente\n        FOREIGN KEY (cliente_id)\n        REFERENCES cliente(cliente_id),\n    CONSTRAINT fk_pagamento_aluguel\n        FOREIGN KEY (aluguel_id)\n        REFERENCES aluguel(aluguel_id)\n) ENGINE = InnoDB;\n\nINSERT INTO pais (pais_id, pais)\nSELECT country_id, country\nFROM sakila.country;\n\nINSERT INTO cidade (cidade_id, cidade, pais_id)\nSELECT city_id, city, country_id\nFROM sakila.city;\n\nINSERT INTO endereco (endereco_id, endereco, cidade_id)\nSELECT address_id, address, city_id\nFROM sakila.address;\n\nINSERT INTO cliente (\n    cliente_id,\n    primeiro_nome,\n    ultimo_nome,\n    email,\n    endereco_id\n)\nSELECT\n    customer_id,\n    first_name,\n    last_name,\n    email,\n    address_id\nFROM sakila.customer;\n\nINSERT INTO ator (\n    ator_id,\n    primeiro_nome,\n    ultimo_nome\n)\nSELECT\n    actor_id,\n    first_name,\n    last_name\nFROM sakila.actor;\n\nINSERT INTO categoria (\n    categoria_id,\n    nome\n)\nSELECT\n    category_id,\n    name\nFROM sakila.category;\n\nINSERT INTO filme (\n    filme_id,\n    titulo,\n    descricao,\n    ano_lancamento,\n    idioma_id,\n    duracao_locacao,\n    preco_da_locacao,\n    duracao_do_filme,\n    custo_substituicao,\n    classificacao\n)\nSELECT\n    film_id,\n    title,\n    description,\n    release_year,\n    language_id,\n    rental_duration,\n    rental_rate,\n    length,\n    replacement_cost,\n    CAST(rating AS CHAR)\nFROM sakila.film;\n\nINSERT INTO filme_categoria (\n    filme_id,\n    categoria_id\n)\nSELECT\n    film_id,\n    category_id\nFROM sakila.film_category;\n\nINSERT INTO filme_ator (\n    ator_id,\n    filme_id\n)\nSELECT\n    actor_id,\n    film_id\nFROM sakila.film_actor;\n\nINSERT INTO inventario (\n    inventario_id,\n    filme_id\n)\nSELECT\n    inventory_id,\n    film_id\nFROM sakila.inventory;\n\nINSERT INTO aluguel (\n    aluguel_id,\n    data_aluguel,\n    inventario_id,\n    cliente_id,\n    data_devolucao\n)\nSELECT\n    rental_id,\n    rental_date,\n    inventory_id,\n    customer_id,\n    return_date\nFROM sakila.rental;\n\nINSERT INTO pagamento (\n    pagamento_id,\n    cliente_id,\n    aluguel_id,\n    valor,\n    data_pagamento\n)\nSELECT\n    payment_id,\n    customer_id,\n    rental_id,\n    amount,\n    payment_date\nFROM sakila.payment;\n\nALTER TABLE filme AUTO_INCREMENT = 2000;\nALTER TABLE pagamento AUTO_INCREMENT = 20000;\n\nSELECT\n    'Base locadora PT-BR criada com sucesso.' AS mensagem;\n\nSHOW TABLES;\n"

setup_path = Path(
    "/content/00_preparar_locadora_sakila_ptbr.sql"
)
setup_path.write_text(
    setup_sql,
    encoding="utf-8"
)

with setup_path.open("r", encoding="utf-8") as entrada:
    resultado = subprocess.run(
        [
            "mariadb",
            "-u",
            "root",
            "--default-character-set=utf8mb4"
        ],
        stdin=entrada,
        text=True,
        capture_output=True
    )

print(resultado.stdout)

if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError(
        "Falha ao criar a base locadora em português."
    )

print("Base locadora PT-BR preparada com sucesso.")


mensagem
Base locadora PT-BR criada com sucesso.
Tables_in_locadora
aluguel
ator
categoria
cidade
cliente
endereco
filme
filme_ator
filme_categoria
inventario
pagamento
pais

Base locadora PT-BR preparada com sucesso.


In [5]:
from pathlib import Path
import subprocess

solucoes_sql = "\n-- =====================================================================\n-- ATIVIDADES PRÁTICAS – BASE SAKILA PT-BR\n-- Consultas, Stored Procedures e Triggers\n--\n-- Renomeie o arquivo conforme o padrão:\n-- NomeSobrenome_Sakila_SQL.sql\n-- =====================================================================\n\nUSE locadora;\n\n-- =====================================================================\n-- ETAPA 0 – PREPARAÇÃO\n-- =====================================================================\n\nSHOW TABLES;\n\nSELECT * FROM filme LIMIT 10;\nSELECT * FROM cliente LIMIT 10;\nSELECT * FROM aluguel LIMIT 10;\n\n\n-- =====================================================================\n-- PARTE I – CONSULTAS INTERMEDIÁRIAS\n-- =====================================================================\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 1 – FILMES E CATEGORIAS\n-- ---------------------------------------------------------------------\n\nSELECT\n    f.filme_id,\n    f.titulo,\n    c.nome AS categoria\nFROM filme AS f\nINNER JOIN filme_categoria AS fc\n    ON fc.filme_id = f.filme_id\nINNER JOIN categoria AS c\n    ON c.categoria_id = fc.categoria_id\nORDER BY\n    f.titulo,\n    c.nome;\n\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 2 – CLIENTES QUE MAIS ALUGARAM\n-- ---------------------------------------------------------------------\n\nSELECT\n    c.cliente_id,\n    CONCAT_WS(\n        ' ',\n        c.primeiro_nome,\n        c.ultimo_nome\n    ) AS cliente,\n    COUNT(a.aluguel_id) AS total_alugueis\nFROM cliente AS c\nINNER JOIN aluguel AS a\n    ON a.cliente_id = c.cliente_id\nGROUP BY\n    c.cliente_id,\n    c.primeiro_nome,\n    c.ultimo_nome\nORDER BY\n    total_alugueis DESC,\n    cliente ASC\nLIMIT 20;\n\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 3 – FILMES LONGOS\n-- ---------------------------------------------------------------------\n\nSELECT\n    filme_id,\n    titulo,\n    duracao_do_filme,\n    preco_da_locacao\nFROM filme\nWHERE duracao_do_filme > 120\nORDER BY\n    duracao_do_filme DESC,\n    titulo ASC;\n\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 4 – QUANTIDADE DE FILMES POR CATEGORIA\n-- ---------------------------------------------------------------------\n\nSELECT\n    c.categoria_id,\n    c.nome AS categoria,\n    COUNT(fc.filme_id) AS quantidade_filmes\nFROM categoria AS c\nLEFT JOIN filme_categoria AS fc\n    ON fc.categoria_id = c.categoria_id\nGROUP BY\n    c.categoria_id,\n    c.nome\nORDER BY\n    quantidade_filmes DESC,\n    c.nome ASC;\n\n\n-- =====================================================================\n-- PARTE II – CONSULTAS AVANÇADAS\n-- =====================================================================\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 5 – ATORES MAIS PRESENTES\n-- ---------------------------------------------------------------------\n\nSELECT\n    a.ator_id,\n    CONCAT_WS(\n        ' ',\n        a.primeiro_nome,\n        a.ultimo_nome\n    ) AS ator,\n    COUNT(fa.filme_id) AS quantidade_filmes\nFROM ator AS a\nINNER JOIN filme_ator AS fa\n    ON fa.ator_id = a.ator_id\nGROUP BY\n    a.ator_id,\n    a.primeiro_nome,\n    a.ultimo_nome\nHAVING COUNT(fa.filme_id) > 25\nORDER BY\n    quantidade_filmes DESC,\n    ator ASC;\n\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 6 – CLIENTES VIP\n-- ---------------------------------------------------------------------\n\nSELECT\n    c.cliente_id,\n    CONCAT_WS(\n        ' ',\n        c.primeiro_nome,\n        c.ultimo_nome\n    ) AS cliente,\n    ROUND(\n        SUM(p.valor),\n        2\n    ) AS gasto_total\nFROM cliente AS c\nINNER JOIN pagamento AS p\n    ON p.cliente_id = c.cliente_id\nGROUP BY\n    c.cliente_id,\n    c.primeiro_nome,\n    c.ultimo_nome\nHAVING SUM(p.valor) > 150.00\nORDER BY\n    gasto_total DESC,\n    cliente ASC;\n\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 7 – FILMES MAIS LUCRATIVOS\n-- ---------------------------------------------------------------------\n\nSELECT\n    f.filme_id,\n    f.titulo,\n    ROUND(\n        SUM(p.valor),\n        2\n    ) AS receita_total,\n    COUNT(DISTINCT a.aluguel_id) AS total_alugueis\nFROM pagamento AS p\nINNER JOIN aluguel AS a\n    ON a.aluguel_id = p.aluguel_id\nINNER JOIN inventario AS i\n    ON i.inventario_id = a.inventario_id\nINNER JOIN filme AS f\n    ON f.filme_id = i.filme_id\nGROUP BY\n    f.filme_id,\n    f.titulo\nORDER BY\n    receita_total DESC,\n    f.titulo ASC\nLIMIT 5;\n\n\n-- =====================================================================\n-- TABELAS AUXILIARES DAS TRIGGERS E DO DESAFIO FINAL\n-- =====================================================================\n\nDROP TRIGGER IF EXISTS trg_auditoria_filme_update;\nDROP TRIGGER IF EXISTS trg_validar_preco_filme_insert;\nDROP TRIGGER IF EXISTS trg_atualizar_resumo_gasto_pagamento;\n\nDROP PROCEDURE IF EXISTS clientes_por_pais;\nDROP PROCEDURE IF EXISTS total_alugueis_cliente;\nDROP PROCEDURE IF EXISTS atualizar_resumo_gasto_clientes;\n\nDROP TABLE IF EXISTS auditoria_filme;\nDROP TABLE IF EXISTS resumo_gasto_cliente;\n\nCREATE TABLE auditoria_filme (\n    auditoria_id BIGINT AUTO_INCREMENT,\n    filme_id INT NOT NULL,\n    titulo_anterior VARCHAR(255),\n    titulo_novo VARCHAR(255),\n    preco_anterior DECIMAL(10,2),\n    preco_novo DECIMAL(10,2),\n    data_alteracao DATETIME NOT NULL\n        DEFAULT CURRENT_TIMESTAMP,\n    tipo_operacao VARCHAR(20) NOT NULL,\n    usuario_banco VARCHAR(100),\n\n    CONSTRAINT pk_auditoria_filme\n        PRIMARY KEY (auditoria_id)\n) ENGINE = InnoDB;\n\nCREATE TABLE resumo_gasto_cliente (\n    cliente_id INT NOT NULL,\n    total_gasto DECIMAL(12,2) NOT NULL\n        DEFAULT 0.00,\n    quantidade_pagamentos INT NOT NULL\n        DEFAULT 0,\n    atualizado_em DATETIME NOT NULL\n        DEFAULT CURRENT_TIMESTAMP,\n\n    CONSTRAINT pk_resumo_gasto_cliente\n        PRIMARY KEY (cliente_id)\n) ENGINE = InnoDB;\n\n\n-- =====================================================================\n-- PARTE III – STORED PROCEDURES\n-- =====================================================================\n\nDELIMITER $$\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 8 – PROCEDURE POR PAÍS\n-- ---------------------------------------------------------------------\n\nCREATE PROCEDURE clientes_por_pais(\n    IN p_nome_pais VARCHAR(50)\n)\nBEGIN\n    SELECT\n        c.cliente_id,\n        CONCAT_WS(\n            ' ',\n            c.primeiro_nome,\n            c.ultimo_nome\n        ) AS cliente,\n        c.email,\n        ci.cidade,\n        p.pais\n    FROM cliente AS c\n    INNER JOIN endereco AS e\n        ON e.endereco_id = c.endereco_id\n    INNER JOIN cidade AS ci\n        ON ci.cidade_id = e.cidade_id\n    INNER JOIN pais AS p\n        ON p.pais_id = ci.pais_id\n    WHERE p.pais = TRIM(p_nome_pais)\n    ORDER BY\n        c.primeiro_nome,\n        c.ultimo_nome;\nEND$$\n\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 9 – QUANTIDADE DE ALUGUÉIS\n-- ---------------------------------------------------------------------\n\nCREATE PROCEDURE total_alugueis_cliente(\n    IN p_cliente_id INT\n)\nBEGIN\n    DECLARE v_cliente_existe INT DEFAULT 0;\n\n    SELECT COUNT(*)\n    INTO v_cliente_existe\n    FROM cliente\n    WHERE cliente_id = p_cliente_id;\n\n    IF v_cliente_existe = 0 THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Cliente não encontrado.';\n    END IF;\n\n    SELECT\n        c.cliente_id,\n        CONCAT_WS(\n            ' ',\n            c.primeiro_nome,\n            c.ultimo_nome\n        ) AS cliente,\n        COUNT(a.aluguel_id) AS total_alugueis\n    FROM cliente AS c\n    LEFT JOIN aluguel AS a\n        ON a.cliente_id = c.cliente_id\n    WHERE c.cliente_id = p_cliente_id\n    GROUP BY\n        c.cliente_id,\n        c.primeiro_nome,\n        c.ultimo_nome;\nEND$$\n\n\n-- ---------------------------------------------------------------------\n-- DESAFIO FINAL – PROCEDURE DE RESUMO DE GASTOS\n-- ---------------------------------------------------------------------\n\nCREATE PROCEDURE atualizar_resumo_gasto_clientes()\nBEGIN\n    DELETE FROM resumo_gasto_cliente;\n\n    INSERT INTO resumo_gasto_cliente (\n        cliente_id,\n        total_gasto,\n        quantidade_pagamentos,\n        atualizado_em\n    )\n    SELECT\n        c.cliente_id,\n        COALESCE(\n            ROUND(SUM(p.valor), 2),\n            0.00\n        ),\n        COUNT(p.pagamento_id),\n        CURRENT_TIMESTAMP\n    FROM cliente AS c\n    LEFT JOIN pagamento AS p\n        ON p.cliente_id = c.cliente_id\n    GROUP BY\n        c.cliente_id;\nEND$$\n\n\n-- =====================================================================\n-- PARTE IV – TRIGGERS\n-- =====================================================================\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 10 – AUDITORIA DE FILMES\n-- OLD = valores anteriores.\n-- NEW = valores posteriores.\n-- ---------------------------------------------------------------------\n\nCREATE TRIGGER trg_auditoria_filme_update\nAFTER UPDATE ON filme\nFOR EACH ROW\nBEGIN\n    INSERT INTO auditoria_filme (\n        filme_id,\n        titulo_anterior,\n        titulo_novo,\n        preco_anterior,\n        preco_novo,\n        data_alteracao,\n        tipo_operacao,\n        usuario_banco\n    ) VALUES (\n        OLD.filme_id,\n        OLD.titulo,\n        NEW.titulo,\n        OLD.preco_da_locacao,\n        NEW.preco_da_locacao,\n        CURRENT_TIMESTAMP,\n        'UPDATE',\n        CURRENT_USER()\n    );\nEND$$\n\n\n-- ---------------------------------------------------------------------\n-- EXERCÍCIO 11 – VALIDAÇÃO DE PREÇO\n-- ---------------------------------------------------------------------\n\nCREATE TRIGGER trg_validar_preco_filme_insert\nBEFORE INSERT ON filme\nFOR EACH ROW\nBEGIN\n    IF NEW.preco_da_locacao IS NULL\n       OR NEW.preco_da_locacao < 1.00 THEN\n\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Preço de locação inválido. O valor mínimo permitido é R$ 1,00.';\n    END IF;\nEND$$\n\n\n-- ---------------------------------------------------------------------\n-- DESAFIO FINAL – TRIGGER DO RESUMO DE GASTOS\n-- ---------------------------------------------------------------------\n\nCREATE TRIGGER trg_atualizar_resumo_gasto_pagamento\nAFTER INSERT ON pagamento\nFOR EACH ROW\nBEGIN\n    INSERT INTO resumo_gasto_cliente (\n        cliente_id,\n        total_gasto,\n        quantidade_pagamentos,\n        atualizado_em\n    ) VALUES (\n        NEW.cliente_id,\n        NEW.valor,\n        1,\n        CURRENT_TIMESTAMP\n    )\n    ON DUPLICATE KEY UPDATE\n        total_gasto =\n            total_gasto + NEW.valor,\n        quantidade_pagamentos =\n            quantidade_pagamentos + 1,\n        atualizado_em =\n            CURRENT_TIMESTAMP;\nEND$$\n\nDELIMITER ;\n\n\n-- =====================================================================\n-- DEMONSTRAÇÃO DAS PROCEDURES\n-- =====================================================================\n\nCALL clientes_por_pais('Brazil');\n\nCALL total_alugueis_cliente(15);\n\n\n-- =====================================================================\n-- DEMONSTRAÇÃO DA AUDITORIA\n-- =====================================================================\n\nSTART TRANSACTION;\n\nUPDATE filme\nSET preco_da_locacao =\n    preco_da_locacao + 0.10\nWHERE filme_id = 1;\n\nSELECT\n    auditoria_id,\n    filme_id,\n    titulo_anterior,\n    titulo_novo,\n    preco_anterior,\n    preco_novo,\n    data_alteracao,\n    tipo_operacao,\n    usuario_banco\nFROM auditoria_filme\nWHERE filme_id = 1\nORDER BY auditoria_id DESC;\n\nROLLBACK;\n\n\n-- =====================================================================\n-- TESTE DA VALIDAÇÃO DE PREÇO\n-- Este comando permanece comentado porque gera erro proposital.\n-- Execute-o separadamente para comprovar a trigger.\n-- =====================================================================\n\n-- INSERT INTO filme (\n--     titulo,\n--     idioma_id,\n--     preco_da_locacao\n-- ) VALUES (\n--     'FILME COM PRECO INVALIDO',\n--     1,\n--     0.50\n-- );\n\n\n-- =====================================================================\n-- DESAFIO FINAL – RESUMO DE GASTOS\n-- =====================================================================\n\nCALL atualizar_resumo_gasto_clientes();\n\nSELECT\n    c.cliente_id,\n    CONCAT_WS(\n        ' ',\n        c.primeiro_nome,\n        c.ultimo_nome\n    ) AS cliente,\n    r.total_gasto,\n    r.quantidade_pagamentos,\n    r.atualizado_em\nFROM cliente AS c\nINNER JOIN resumo_gasto_cliente AS r\n    ON r.cliente_id = c.cliente_id\nWHERE r.total_gasto > 150.00\nORDER BY\n    r.total_gasto DESC,\n    cliente ASC;\n\n\n-- =====================================================================\n-- VERIFICAÇÃO DOS OBJETOS CRIADOS\n-- =====================================================================\n\nSHOW PROCEDURE STATUS\nWHERE Db = 'locadora'\n  AND Name IN (\n      'clientes_por_pais',\n      'total_alugueis_cliente',\n      'atualizar_resumo_gasto_clientes'\n  );\n\nSHOW TRIGGERS\nFROM locadora;\n\n\n-- =====================================================================\n-- EXPLICAÇÃO DO DESAFIO FINAL\n--\n-- A procedure utiliza JOIN e GROUP BY para consolidar o total gasto e\n-- a quantidade de pagamentos de cada cliente.\n--\n-- A trigger mantém o resumo atualizado após cada novo pagamento.\n--\n-- Em um sistema real, a solução pode alimentar painéis gerenciais,\n-- programas de fidelidade, campanhas e identificação de clientes VIP.\n-- =====================================================================\n"

arquivo_solucoes = Path(
    "/content/NomeSobrenome_Sakila_SQL.sql"
)
arquivo_solucoes.write_text(
    solucoes_sql,
    encoding="utf-8"
)

with arquivo_solucoes.open(
    "r",
    encoding="utf-8"
) as entrada:
    resultado = subprocess.run(
        [
            "mariadb",
            "-u",
            "root",
            "--default-character-set=utf8mb4"
        ],
        stdin=entrada,
        text=True,
        capture_output=True
    )

print(resultado.stdout)

if resultado.returncode != 0:
    print("ERRO DO MYSQL/MARIADB:")
    print(resultado.stderr)
    raise RuntimeError(
        "Falha ao executar as soluções da atividade."
    )

print(
    "Consultas, procedures e triggers executadas com sucesso."
)
print(
    "Arquivo SQL criado em:",
    arquivo_solucoes
)


Tables_in_locadora
aluguel
ator
auditoria_filme
categoria
cidade
cliente
endereco
filme
filme_ator
filme_categoria
inventario
pagamento
pais
resumo_gasto_cliente
filme_id	titulo	descricao	ano_lancamento	idioma_id	duracao_locacao	preco_da_locacao	duracao_do_filme	custo_substituicao	classificacao
1	ACADEMY DINOSAUR	A Epic Drama of a Feminist And a Mad Scientist who must Battle a Teacher in The Canadian Rockies	2006	1	6	0.99	86	20.99	PG
2	ACE GOLDFINGER	A Astounding Epistle of a Database Administrator And a Explorer who must Find a Car in Ancient China	2006	1	3	4.99	48	12.99	G
3	ADAPTATION HOLES	A Astounding Reflection of a Lumberjack And a Car who must Sink a Lumberjack in A Baloon Factory	2006	1	7	2.99	50	18.99	NC-17
4	AFFAIR PREJUDICE	A Fanciful Documentary of a Frisbee And a Lumberjack who must Chase a Monkey in A Shark Tank	2006	1	5	2.99	117	26.99	G
5	AFRICAN EGG	A Fast-Paced Documentary of a Pastry Chef And a Dentist who must Pursue a Forensic Psychologist in The Gulf of Mexico	2006

In [6]:
import subprocess

consultas_verificacao = "\nUSE locadora;\n\nSELECT\n    'CONTAGEM DAS TABELAS PRINCIPAIS' AS etapa;\n\nSELECT 'Filmes' AS tabela, COUNT(*) AS total FROM filme\nUNION ALL\nSELECT 'Clientes', COUNT(*) FROM cliente\nUNION ALL\nSELECT 'Aluguéis', COUNT(*) FROM aluguel\nUNION ALL\nSELECT 'Pagamentos', COUNT(*) FROM pagamento\nUNION ALL\nSELECT 'Atores', COUNT(*) FROM ator\nUNION ALL\nSELECT 'Categorias', COUNT(*) FROM categoria;\n\nSELECT\n    'TOP 5 FILMES MAIS LUCRATIVOS' AS etapa;\n\nSELECT\n    f.titulo,\n    ROUND(SUM(p.valor), 2) AS receita_total,\n    COUNT(DISTINCT a.aluguel_id) AS total_alugueis\nFROM pagamento AS p\nINNER JOIN aluguel AS a\n    ON a.aluguel_id = p.aluguel_id\nINNER JOIN inventario AS i\n    ON i.inventario_id = a.inventario_id\nINNER JOIN filme AS f\n    ON f.filme_id = i.filme_id\nGROUP BY\n    f.filme_id,\n    f.titulo\nORDER BY\n    receita_total DESC,\n    f.titulo\nLIMIT 5;\n\nSELECT\n    'PROCEDURES CRIADAS' AS etapa;\n\nSHOW PROCEDURE STATUS\nWHERE Db = 'locadora';\n\nSELECT\n    'TRIGGERS CRIADAS' AS etapa;\n\nSHOW TRIGGERS\nFROM locadora;\n"

resultado = subprocess.run(
    [
        "mariadb",
        "-u",
        "root",
        "--table",
        "--default-character-set=utf8mb4",
        "-e",
        consultas_verificacao
    ],
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError(
        "Erro durante a verificação final."
    )

print("Atividade concluída.")


+---------------------------------+
| etapa                           |
+---------------------------------+
| CONTAGEM DAS TABELAS PRINCIPAIS |
+---------------------------------+
+------------+-------+
| tabela     | total |
+------------+-------+
| Filmes     |  1000 |
| Clientes   |   599 |
| Aluguéis   | 16044 |
| Pagamentos | 16044 |
| Atores     |   200 |
| Categorias |    16 |
+------------+-------+
+------------------------------+
| etapa                        |
+------------------------------+
| TOP 5 FILMES MAIS LUCRATIVOS |
+------------------------------+
+-------------------+---------------+----------------+
| titulo            | receita_total | total_alugueis |
+-------------------+---------------+----------------+
| TELEGRAPH VOYAGE  |        231.73 |             27 |
| WIFE TURN         |        223.69 |             31 |
| ZORRO ARK         |        214.69 |             31 |
| GOODFELLAS SALUTE |        209.69 |             31 |
| SATURDAY LAMBS    |        204.72 |   

## Baixar o SQL final
Antes da entrega, renomeie o arquivo conforme o padrão solicitado pelo professor.

In [7]:
from google.colab import files

# Renomeie o arquivo antes da entrega, substituindo NomeSobrenome.
files.download(
    "/content/NomeSobrenome_Sakila_SQL.sql"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>